**Download the SraRunTable.txt file from NCBI:**

This file contains the list of accessions (sequencing runs) 

In [ ]:
%%bash

time {

OUTDIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363"
mkdir -p "$OUTDIR"

rm -f "$OUTDIR/SraRunTable.txt"

curl --http1.1 -L --retry 5 --retry-delay 10 \
  "https://trace.ncbi.nlm.nih.gov/Traces/sra-db-be/runinfo?acc=SRP093363" \
  -o "$OUTDIR/SraRunTable.txt"

ls -lh "$OUTDIR/SraRunTable.txt"
head -5 "$OUTDIR/SraRunTable.txt"

}

**Display in a Table, the headings row and first two rows of data:**

Count across to the 'Samples' column starting at column zero first. In this case the 'Samples' column is column 24.

In [ ]:
%%time 

import pandas as pd

OUTDIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363"

# Show ALL columns
pd.set_option('display.max_columns', None)

# Optional: widen display
pd.set_option('display.width', None)

# Load file
df = pd.read_csv(f"{OUTDIR}/SraRunTable.txt")

# Display first 2 rows
df.head(2)

**Display the contents of column 0, column 24 and column 29 only:**

There are 12 SRS sample files in this project, however we are only interested in the samples 

In [ ]:
%%time 

import pandas as pd

OUTDIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363"

# Load the file
df = pd.read_csv(f"{OUTDIR}/SraRunTable.txt")

# Display column 0 (Run), column 24 (Sample) and 29 (SampleName) only
df.iloc[:, [0,24,29]]

**Get accession RUN numbers of interest:**

As can be seen above, there are 12 sample files in this project. Each of these files is between 3 and 4 GB in size. It can take from minutes to hours to download files of this size, depending on your interent speed. Therefore, we only download files which we are interested in. In this case we are interested in samples related to "High-Fat Diet Control" and "High-Fat Diet Tumor". Here we extract those accession RUN numbers and put them into a new file named finalsamples.txt.

In [ ]:
%%time 

import pandas as pd

OUTDIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363"

# Load the file
df = pd.read_csv(f"{OUTDIR}/SraRunTable.txt")

# -----------------------------
# High-Fat Diet Control
# -----------------------------
control = df[
    df.iloc[:, 29].astype(str).str.contains(
        "High-Fat Diet Control",
        case=False,
        na=False
    )
# ].iloc[:, [0, 24, 29]]
].iloc[:, [0]] # Only keep Run data

# Save first dataset
control.to_csv(
    f"{OUTDIR}/finalsamples.txt",
    index=False,
    header=False
)

# -----------------------------
# High-Fat Diet Tumor
# -----------------------------
tumor = df[
    df.iloc[:, 29].astype(str).str.contains(
        "High-Fat Diet Tumor",
        case=False,
        na=False
    )
# ].iloc[:, [0, 24, 29]]
].iloc[:, [0]] # Only keep Run data

# Append to same file
tumor.to_csv(
    f"{OUTDIR}/finalsamples.txt",
    mode='a',
    index=False,
    header=False
)

# Display both
display(control)
display(tumor)

**Download the SRA RUN files of interest based on the 'finalsamples.txt' file contents:**

In [ ]:
%%bash

time {

OUTDIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363"

cut -d',' -f1 "$OUTDIR/finalsamples.txt" | while read SRR; do

    # Clean hidden characters/spaces
    SRR=$(echo "$SRR" | tr -d '\r' | xargs)

    # Skip empty lines
    [ -z "$SRR" ] && continue

    # Expected SRA file path
    SRA_FILE="$OUTDIR/$SRR/$SRR.sra"

    # Skip if already downloaded
    if [ -f "$SRA_FILE" ]; then
        echo "Skipping $SRR (already exists)"
        continue
    fi

    echo "Downloading $SRR ..."

    prefetch "$SRR" --output-directory "$OUTDIR/sra"

done
}

**Convert SRA files to FASTQ files:**

Note: An SRA file of ~4GB will convert to fastq file ~13GB

In [ ]:
%%bash

time {

SRA_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/sra"
FASTQ_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/fastq"

mkdir -p "$FASTQ_DIR"

find "$SRA_DIR" -name "*.sra" | while read FILE; do

    BASENAME=$(basename "$FILE" .sra)

    FASTQ1="$FASTQ_DIR/${BASENAME}_1.fastq"
    FASTQ2="$FASTQ_DIR/${BASENAME}_2.fastq"
    FASTQ_SINGLE="$FASTQ_DIR/${BASENAME}.fastq"

    # Skip if FASTQ already exists
    if [ -f "$FASTQ1" ] || [ -f "$FASTQ_SINGLE" ]; then
        echo "Skipping $BASENAME (FASTQ already exists)"
        continue
    fi

    echo "Converting $BASENAME ..."

    fasterq-dump "$FILE" \
        --split-files \
        --threads 4 \
        --outdir "$FASTQ_DIR"
}
done


**Delete all of the sra files and folders located inside the sra folder:**

This will free up about 3 to 4 GB per SRA file.

In [ ]:
%%bash

time {
    
rm -rf "/var/www/html/usb/jupyter/my_projects/data/SRP093363/sra"/SRR*
    
}

**gzip the output FASTQ files:**

Note: A fastq file of ~13GB will be compressed to a fastq.gz file of ~4GB

In [ ]:
%%bash

FASTQ_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/fastq"

# Change the dir name to:
# FASTQ_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/fastq_gz"

for file in "$FASTQ_DIR"/*.fastq; do

    # Skip if no .fastq files exist
    [ -e "$file" ] || continue

    gz_file="${file}.gz"

    if [ -f "$gz_file" ]; then
        echo "Skipping: $(basename "$file") -> .gz already exists"
    else
        echo "Compressing: $(basename "$file")"
        gzip "$file"
    fi

done

echo "Finished."

**Delete the original .fastq files that have matching .fastq.gz files:**

In [ ]:
%%bash

FASTQ_DIR="/var/www/html/usb/jupyter/my_projects/data/SRP093363/fastq"

for file in "$FASTQ_DIR"/*.fastq; do
    [ -e "$file" ] || continue

    if [ -f "${file}.gz" ]; then
        echo "Deleting: $(basename "$file")"
        rm "$file"
    fi
done

echo "Finished."